# Dialforge Cloud Benchmark

Runs the Dialforge AI stack on a cloud GPU instead of your PC. It benchmarks **Qwen 3 1.7B / 4B / 8B**, **faster-whisper small.en**, **Chatterbox Nano**, and synthetic **STT → LLM → TTS** turns.

Before running: in Colab choose **Runtime → Change runtime type → GPU**. Then use **Runtime → Run all**. No SIP credentials are needed and no real phone calls are made.

In [ ]:
# Benchmark controls. The defaults test all three Dialforge Qwen tiers.
QWEN_MODELS = ['qwen3:1.7b', 'qwen3:4b', 'qwen3:8b']
QWEN_REPEATS = 3
COMPONENT_REPEATS = 3
PIPELINE_TURNS = 5
PIPELINE_MODEL = 'qwen3:4b'
print('Models:', ', '.join(QWEN_MODELS))

In [ ]:
# Verify that Colab actually assigned a GPU.
import shutil, subprocess
if not shutil.which('nvidia-smi'):
    raise RuntimeError('No NVIDIA GPU is attached. In Colab choose Runtime > Change runtime type > GPU, then reconnect.')
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
# Install the same core components used by Dialforge.
import subprocess, sys
subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'psutil==7.0.0', 'requests==2.32.5', 'numpy',
                'faster-whisper==1.2.0',
                'git+https://github.com/resemble-ai/chatterbox.git@5de7a54aa4e5e2baadb0182dde554908b48b85c2'], check=True)
print('Dependencies installed.')

In [ ]:
# Start Ollama and fetch the public benchmark runner.
import os, pathlib, subprocess, time, urllib.request, requests
OLLAMA_LOG = open('/content/ollama.log', 'w')
ollama_proc = subprocess.Popen(['ollama', 'serve'], stdout=OLLAMA_LOG, stderr=subprocess.STDOUT)
for _ in range(45):
    try:
        if requests.get('http://127.0.0.1:11434/api/tags', timeout=2).ok:
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError('Ollama failed to start. See /content/ollama.log')
runner = '/content/dialforge_colab_benchmark.py'
urllib.request.urlretrieve('https://raw.githubusercontent.com/SumamaAhmed69/Axemetric-Caller-Beta-Runtime/main/benchmarks/dialforge_colab_benchmark.py', runner)
print('Ollama ready. Benchmark runner downloaded.')

In [ ]:
# Run the benchmark. Model files stay on this temporary Colab machine.
import subprocess, sys
cmd = [sys.executable, '/content/dialforge_colab_benchmark.py',
       '--models', *QWEN_MODELS,
       '--qwen-repeats', str(QWEN_REPEATS),
       '--component-repeats', str(COMPONENT_REPEATS),
       '--pipeline-turns', str(PIPELINE_TURNS),
       '--pipeline-model', PIPELINE_MODEL,
       '--output-dir', '/content/dialforge-benchmark']
subprocess.run(cmd, check=True)

In [ ]:
# Display the finished report inside Colab.
from IPython.display import HTML, display
report_html = '/content/dialforge-benchmark/dialforge-benchmark-report.html'
display(HTML(open(report_html, encoding='utf-8').read()))

In [ ]:
# Optional: download the small reports to your PC. This downloads results only, not the AI models.
# Remove the # characters below if you want copies.
# from google.colab import files
# files.download('/content/dialforge-benchmark/dialforge-benchmark-report.html')
# files.download('/content/dialforge-benchmark/dialforge-benchmark-report.json')